# 02 · Noise and simulation methods / Noise နှင့် simulation mode များ

**Goal:** choose statevector, density matrix or trajectories. About 15 minutes.
- Statevector: noiseless pure-state evolution.
- Density matrix: deterministic exact channel evolution, subject to floating-point error.
- Trajectory: a sampled noise ensemble; increasing shots does not increase its histories.

Exact noise model နှင့် sampled estimate ကို ခွဲခြားပါ။ Trajectory အရေအတွက် ပိုများလျှင်
Monte Carlo error ကို လျှော့ချနိုင်ပါသည်။


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import nexqus
from nexqus import Circuit, Simulator, Parameter, Noise, NoiseModel, algorithms
assert nexqus.__version__ == '1.4.0', 'Install NexQus SDK 1.4.0 in this kernel environment and restart it'
print('NexQus SDK:', nexqus.__version__)


In [ ]:
flipped = Simulator().run(Circuit(1).noise(Noise.bit_flip(.2), 0), shots=0)
np.testing.assert_allclose(flipped.probabilities, [.8, .2], atol=1e-12)
print('Bit-flip p=0.2:', flipped.probabilities)


In [ ]:
q = algorithms.bell()
model = NoiseModel(Noise.depolarizing(.08), Noise.amplitude_damping(.04))
ideal = Simulator().run(q, shots=0)
exact = Simulator(method='density_matrix').run(q, noise=model, shots=0)
sampled = Simulator(method='trajectory').run(q, noise=model, trajectories=2048,
                                           batch_size=64, shots=0, seed=42)
tvd = float(np.abs(exact.probabilities - sampled.probabilities).sum()/2)
print('Exact:', exact.probabilities)
print('Trajectory estimate:', sampled.probabilities)
print('Total variation distance:', tvd)
assert tvd < .08
assert np.isclose(np.trace(exact.density_matrix), 1)
assert np.linalg.eigvalsh(exact.density_matrix).min() > -1e-12


In [ ]:
positions = np.arange(4)
plt.figure(figsize=(7, 3))
for offset, (name, probabilities) in enumerate([('Ideal', ideal.probabilities), ('Exact noise', exact.probabilities), ('Trajectories', sampled.probabilities)]):
    plt.bar(positions + (offset-1)*.25, probabilities, width=.25, label=name)
plt.xticks(positions, ['00', '01', '10', '11']); plt.ylabel('Probability')
plt.title('Specified noise model: exact versus sampled'); plt.legend()
plt.tight_layout(); plt.show()


Global noise rules act after each gate on every gate-touched wire. Depolarizing uses
E(rho)=(1-p)rho+p I/2 on the selected qubit. Reset and intermediate measurement are ideal
unless explicit noise is inserted. Do not assume every library uses the same p convention.

Global noise သည် gate လုပ်ပြီးတိုင်း သက်ဆိုင်ရာ qubit များတွင် ထည့်ပါသည်။


In [ ]:
# Storage estimates only: do not allocate these large density matrices.
for n in [10, 16, 20]:
    state_mib = 16 * 2**n / 1024**2
    density_mib = 16 * 4**n / 1024**2
    print(f'{n} qubits: statevector {state_mib:g} MiB; density matrix {density_mib:g} MiB')
try:
    Simulator(method='density_matrix', max_memory_mb=16).run(Circuit(12), shots=0)
except MemoryError:
    print('Expected: the working-memory guard blocked this allocation.')
else:
    raise AssertionError('Memory guard should reject this run')


**Try it:** compare 256 and 2048 trajectories with a fixed seed. One trial need not improve monotonically;
sample more seeds to assess variation. Peak process memory exceeds the main-state estimates above.
